# Module 12 - 실습 1: 체크포인팅 (MemorySaver)

## 학습 목표
- `MemorySaver`로 그래프 실행 상태를 저장할 수 있다
- `thread_id`로 대화 이력을 관리할 수 있다
- 체크포인트로 마지막 상태를 조회할 수 있다

## 참조 자료
- 학습 문서: `docs/part5-advanced/12-langgraph-advanced.md` (섹션 2.1)

## 개념 요약

**체크포인팅의 필요성**:
```
체크포인팅 없이:
  [A] → [B] → [C] → [D] ← 장애!
  → 전체 재시작 필요 (비용 낭비)

체크포인팅 있으면:
  [A] → ✓저장 → [B] → ✓저장 → [C] → ✓저장 → [D] ← 장애!
  → [C] 체크포인트에서 재개!
```

**thread_id**: 실행 세션을 식별하는 고유 키
- 같은 thread_id → 이전 체크포인트에서 재개
- 새 thread_id → 처음부터 새로 실행

## 체크포인터 종류 비교

| 체크포인터 | 저장소 | 영속성 | 사용 시나리오 |
|-----------|--------|--------|-------------|
| **MemorySaver** | 메모리 (dict) | 프로세스 종료 시 소멸 | 개발/테스트, 단일 세션 |
| **SqliteSaver** | SQLite 파일 | 파일 기반 영속 | 소규모 프로덕션, 단일 서버 |
| **PostgresSaver** | PostgreSQL | DB 기반 완전 영속 | 대규모 프로덕션, 분산 환경 |

> 이 실습에서는 MemorySaver를 사용합니다 (설치 없이 바로 사용 가능).

In [ ]:
import sys
sys.path.insert(0, '..')

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from common.fake_llm import FakeLLM

print("체크포인팅 실습 준비 완료")

## 실습 1: ProcessState 정의

3단계 처리 파이프라인의 상태를 정의하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: input_data: str - 입력 데이터
# 힌트 2: step1_result, step2_result, step3_result: str | None
# 힌트 3: current_step: str - 현재 단계 추적

class ProcessState(TypedDict):
    pass  # TODO: 5개 필드 정의

## 실습 2: 3단계 처리 그래프 구현

step1, step2, step3 노드를 가진 그래프를 구현하고 MemorySaver를 연결하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: 각 노드는 이전 단계 결과를 참조하여 결과를 생성
# 힌트 2: graph.compile(checkpointer=MemorySaver()) 로 체크포인터 연결

def step1_node(state: ProcessState) -> dict:
    """1단계: 데이터 수집."""
    print("  [Step 1] 데이터 수집 중...")
    # TODO: step1_result와 current_step 반환
    pass

def step2_node(state: ProcessState) -> dict:
    """2단계: 데이터 분석."""
    print("  [Step 2] 데이터 분석 중...")
    # TODO: step2_result와 current_step 반환
    pass

def step3_node(state: ProcessState) -> dict:
    """3단계: 결과 생성."""
    print("  [Step 3] 결과 생성 중...")
    # TODO: step3_result와 current_step 반환
    pass

# 그래프 구성
# TODO: 그래프 생성, 노드 추가, 엣지 연결, MemorySaver 연결
checkpointer = MemorySaver()
# app = ...

## 실습 3: thread_id로 실행 및 상태 조회

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: config = {"configurable": {"thread_id": "task-001"}}
# 힌트 2: app.invoke(initial_state, config=config) 로 실행
# 힌트 3: app.get_state(config) 로 마지막 상태 조회

config = {"configurable": {"thread_id": "task-001"}}

initial_state = {
    "input_data": "사용자 로그 데이터",
    "step1_result": None,
    "step2_result": None,
    "step3_result": None,
    "current_step": "start",
}

print("그래프 실행 시작...")
# TODO: app.invoke() 호출
result = None

print(f"\n최종 결과: {result['step3_result'] if result else 'N/A'}")

In [ ]:
# 체크포인트 상태 조회
# TODO: app.get_state(config)로 상태 조회
state_snapshot = None

if state_snapshot:
    print(f"마지막 단계: {state_snapshot.values.get('current_step', 'N/A')}")

In [ ]:
# 검증 셀
assert result is not None, "app.invoke()를 구현하세요"
assert result.get("step3_result") is not None, "step3_result가 None입니다"
assert result.get("current_step") == "step3", f"마지막 단계가 'step3'이어야 합니다. 현재: {result.get('current_step')}"
print("✅ 실습 3 완료! 3단계 파이프라인이 체크포인팅과 함께 실행되었습니다.")

## 실습 4: 다른 thread_id로 새 실행

다른 `thread_id`를 사용하면 독립적인 새 실행이 시작됩니다.

In [ ]:
# TODO: 새 thread_id로 실행
# 힌트: config2 = {"configurable": {"thread_id": "task-002"}}

config2 = {"configurable": {"thread_id": "task-002"}}

initial_state2 = {
    "input_data": "다른 사용자의 데이터",
    "step1_result": None,
    "step2_result": None,
    "step3_result": None,
    "current_step": "start",
}

print("새 thread_id로 실행...")
# TODO: 실행
result2 = None

In [ ]:
# 검증 셀 - 두 실행이 독립적임을 확인
assert result2 is not None, "두 번째 실행을 구현하세요"
# thread_id-001의 결과는 영향받지 않아야 함
state1 = app.get_state({"configurable": {"thread_id": "task-001"}})
state2 = app.get_state({"configurable": {"thread_id": "task-002"}})
assert state1.values.get("input_data") == "사용자 로그 데이터"
assert state2.values.get("input_data") == "다른 사용자의 데이터"
print("✅ 실습 4 완료! 두 thread_id가 독립적으로 관리됩니다.")

## 정리

이번 실습에서 배운 내용:
1. `MemorySaver`로 그래프 실행마다 상태를 저장한다
2. `thread_id`로 실행 세션을 구분한다 (같은 ID → 재개, 다른 ID → 새 실행)
3. `app.get_state(config)`로 언제든지 상태를 조회할 수 있다

## 다음 모듈
- **실습 2**: `02_subgraphs.ipynb` - 서브그래프 패턴